Preparations:
- load prediction.csv
    - drop the lines with the non-used ifs
    - if there is more then one if, take mean

- add column "sample" to master table
- join them
- drop unnecessary columns (matadata, paths, ranks)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import ast
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score, precision_recall_curve, confusion_matrix

# Configuration
CSV_PATH =  "/home/bri/pymol/data/predictions/preds_snir.csv" 
MASTER_PATH =  "/home/bri/pymol/data/3ab/table/master_table_snir.csv" 
YAML_PATH = "/home/bri/pymol/EDA/static/metric_directions.yaml"
OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/snir_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


KEEP_INTERFACES = {"A,C", "B,C"}   # as option if there is more than one, default A,B



# Load data
df_if = pd.read_csv(CSV_PATH)
df_if.info()


<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Columns: 138 entries, sample to esmfold_mpnn_nll_b
dtypes: bool(1), float64(127), str(10)
memory usage: 218.5 KB


#### all

In [2]:
num_cols = df_if.select_dtypes(include=['number']).columns
other_cols = [c for c in df_if.columns if c not in num_cols and c != 'sample']

agg_dict = {col: 'mean' for col in num_cols}
agg_dict.update({col: 'first' for col in other_cols})

df_1if = df_if.groupby('sample').agg(agg_dict).reset_index()

df_1if

/tmp/ipykernel_5272/4193676620.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_1if = df_if.groupby('sample').agg(agg_dict).reset_index()


,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,binding,interface,chai_unconstrained_model_path,chai_constrained_model_path,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm,boltz_free_model_path,boltz_template_model_path
0,5ifj_abc_cut,2.593883,95.404666,28.410000,2.370000,1.988426,95.031040,0.824021,0.321900,0.872133,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9454320669174194, 0.9492036700248718, 0.17...","[[[0.9454320669174194, 0.8899917602539062, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
1,5ifj_abc_cut_A_D111A,2.815847,95.282854,57.816667,3.936667,2.304955,94.104966,0.806316,0.302767,0.856633,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.9454193711280823, 0.9488046169281006, 0.17...","[[[0.9454193711280823, 0.8907997608184814, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
2,5ifj_abc_cut_A_K110A,2.790728,94.871984,36.826667,3.240000,2.416944,92.917901,0.810150,0.296900,0.830000,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9449247717857361, 0.9489190578460693, 0.17...","[[[0.9449247717857361, 0.889130175113678, 0.07...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
3,5ifj_abc_cut_A_S64A,2.539446,95.613237,34.810000,2.686667,1.973646,95.207555,0.818178,0.312200,0.889533,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.945385217666626, 0.948758602142334, 0.1724...","[[[0.945385217666626, 0.8885297775268555, 0.07...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
4,5ifj_abc_cut_A_W52A,3.005329,94.814333,45.003333,4.813333,2.193750,92.632748,0.755118,0.311567,0.809633,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9452540874481201, 0.9489129781723022, 0.17...","[[[0.9452540874481201, 0.8904703855514526, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
5,5ifj_abc_cut_A_Y66A,2.701376,95.337838,32.103333,2.763333,2.138492,94.162554,0.805907,0.308767,0.858167,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9473956227302551, 0.9488828778266907, 0.16...","[[[0.9473956227302551, 0.8850024938583374, 0.0...","[[0.9453608989715576, 0.9489309191703796, 0.17...","[[[0.9453608989715576, 0.8903481960296631, 0.0...",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...
6,5ifj_abc_cut_B_L116A,2.806607,95.032996,30.606667,2.200000,2.177498,94.090785,0.793489,0.295733,0.844567,...,False,"A,B",/scicore/home/schwede/barta0000/scoring_pipeli...,/scicore/home/schwede/barta0000/scoring_pipeli...,"[[0.9485109448432922, 0.9499985575675964, 0.17...","[[[0.9485109448432922, 0.8920019865036011, 0.0...","[[0.9452025294303894, 0.9490475058555603, 0.17...","[[[0.9452025294303894, 0.8905109763145447, 0.0...",/s

In [3]:
df_2if = df_1if.drop(columns=["binding", "interface","chai_unconstrained_model_path", "chai_constrained_model_path", "boltz_free_model_path", "boltz_template_model_path"])
df_2if

,sample,af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,af3_pdockq,af3_pdockq2,...,esmfold_esm3dg_dg_b,esmfold_esm3dg_dg_std,esmfold_mpnn_confidence,esmfold_mpnn_nll,esmfold_mpnn_nll_a,esmfold_mpnn_nll_b,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm
0,5ifj_abc_cut,2.593883,95.404666,28.410000,2.370000,1.988426,95.031040,0.824021,0.321900,0.872133,...,3.138038,0.218290,0.188899,1.707462,1.309943,2.104981,"[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9454320669174194, 0.9492036700248718, 0.17...","[[[0.9454320669174194, 0.8899917602539062, 0.0..."
1,5ifj_abc_cut_A_D111A,2.815847,95.282854,57.816667,3.936667,2.304955,94.104966,0.806316,0.302767,0.856633,...,3.096884,0.224287,0.193141,1.675427,1.333623,2.017231,"[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.9454193711280823, 0.9488046169281006, 0.17...","[[[0.9454193711280823, 0.8907997608184814, 0.0..."
2,5ifj_abc_cut_A_K110A,2.790728,94.871984,36.826667,3.240000,2.416944,92.917901,0.810150,0.296900,0.830000,...,3.015681,0.209554,0.194116,1.674882,1.299580,2.050184,"[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9449247717857361, 0.9489190578460693, 0.17...","[[[0.9449247717857361, 0.889130175113678, 0.07..."
3,5ifj_abc_cut_A_S64A,2.539446,95.613237,34.810000,2.686667,1.973646,95.207555,0.818178,0.312200,0.889533,...,3.052398,0.213692,0.190961,1.699475,1.282060,2.116890,"[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.945385217666626, 0.948758602142334, 0.1724...","[[[0.945385217666626, 0.8885297775268555, 0.07..."
4,5ifj_abc_cut_A_W52A,3.005329,94.814333,45.003333,4.813333,2.193750,92.632748,0.755118,0.311567,0.809633,...,3.156102,0.236498,0.185690,1.730604,1.315515,2.145692,"[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9452540874481201, 0.9489129781723022, 0.17...","[[[0.9452540874481201, 0.8904703855514526, 0.0..."
5,5ifj_abc_cut_A_Y66A,2.701376,95.337838,32.103333,2.763333,2.138492,94.162554,0.805907,0.308767,0.858167,...,3.205366,0.209332,0.185966,1.727805,1.309762,2.145849,"[[0.9473956227302551, 0.9488828778266907, 0.16...","[[[0.9473956227302551, 0.8850024938583374, 0.0...","[[0.9453608989715576, 0.9489309191703796, 0.17...","[[[0.9453608989715576, 0.8903481960296631, 0.0..."
6,5ifj_abc_cut_B_L116A,2.806607,95.032996,30.606667,2.200000,2.177498,94.090785,0.793489,0.295733,0.844567,...,3.035983,0.208600,0.182653,1.745321,1.322374,2.168267,"[[0.9485109448432922, 0.9499985575675964, 0.17...","[[[0.9485109448432922, 0.8920019865036011, 0.0...","[[0.9452025294303894, 0.9490475058555603, 0.17...","[[[0.9452025294303894, 0.8905109763145447, 0.0..."
7,5ifj_abc_cut_B_V38A,2.714935,95.220947,37.886667,2.943333,2.238356,93.849034,0.803100,0.308100,0.858500,...,3.181652,0.208299,0.188240,1.712567,1.312490,2.112644,"[[0.9468277096748352, 0.9472519159317017, 0.10...","[[[0.9468277096748352, 0.8879522085189819, 0.0...","[[0.945318341255188, 0.9495519995689392, 0.177...","[[[0.945318341255188, 0.890434980392456, 0.072..."
8,5ifj_abc_cut_B_Y37A,3.004751,94.723352,19.743333,1.660000,2.273611,92.894702,0.721584,0.300000,0.815433,...,3.144416,0.244345,0.196540,1.665227,1.292638,2.037816,"[[0.940564751625061, 0.9448128938674927, 0.095...","[[[0.940564751625061, 0.8819113969802856, 0.04...","[[0.9450069665908813, 0.94929438829422, 0.1781...","[[[0.9450069665908813, 0.889803946018219, 0.07..."
9,5ifj_abc_cut_all_A,3.597543,93.235998,39.330000,4.260000,2.361091,86.447573,0.413262,0.275333,0.470667,...,2.979492,0.273020,0.193412,1.692783,1.275496,2.110071,"[[0.9467074275016785, 0.9420090913772583, 0.11...","[[[0.9467074275016785, 0.8753525018692017, 0.0...",

## master df

##### snir


In [5]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')
master_df

,source,type,sample,fab,interface,vh/vl,target,target_residues,target_sequence,binder,KD[M],notes
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_E,NaN,PLQPEQPFP,1,0.000010,Fab 1E03 + KD[M] approximated from Snir et. al...
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...,gliadin_alpha_Q,NaN,PLQPQQPFP,1,0.000100,Fab 1E01 + KD[M] approximated from Snir et. al...
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,1,0.000001,Fab 1E01 + KD[M] approximated from Snir et. al...
...,...,...,...,...,...,...,...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLYSSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,10.0,"B,C",DIVMTQSPDSLAVSLGERATINCKSSQSVLASSNNKNYLAWYQQKP...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,10.0,"A,C",EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...,gliadin_gamma_E,NaN,LQPEQPFP,0,NaN,NaN


In [6]:
master_df2 = master_df.drop(columns=["interface", "fab", "vh/vl", "target", "target_residues", "target_sequence", "notes"])
master_df2

,source,type,sample,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1,0.000010
1,snir_ab,original,5ifj_abc_cut,1,0.000010
2,snir_ab,original,5ig7_abc_cut,1,0.000100
3,snir_ab,original,5ig7_abc_cut,1,0.000100
4,snir_ab,original,5ijk_acx_cut,1,0.000001
...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,0,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,0,NaN


In [7]:
master_df3 = master_df2.drop_duplicates(keep="last")
master_df3

,source,type,sample,binder,KD[M]
1,snir_ab,original,5ifj_abc_cut,1,0.000010
3,snir_ab,original,5ig7_abc_cut,1,0.000100
5,snir_ab,original,5ijk_acx_cut,1,0.000001
7,snir_ab,alanine_scan,5ifj_abc_cut_A_D111A,0,NaN
9,snir_ab,alanine_scan,5ifj_abc_cut_A_K110A,0,NaN
11,snir_ab,alanine_scan,5ifj_abc_cut_A_S64A,0,NaN
13,snir_ab,alanine_scan,5ifj_abc_cut_A_W52A,0,NaN
15,snir_ab,alanine_scan,5ifj_abc_cut_A_Y66A,0,NaN
17,snir_ab,alanine_scan,5ifj_abc_cut_all_A,0,NaN
19,snir_ab,alanine_scan,5ifj_abc_cut_B_L116A,0,NaN


##### mcmahon / germinal

In [8]:

master_df= pd.read_csv(MASTER_PATH, delimiter=',')
#master_df = master.drop(columns=["Unnamed: 11", "Unnamed: 12", "Unnamed: 13", "Unnamed: 14", "Unnamed: 15", "Unnamed: 16", "Unnamed: 17"])
master_df.head()

master_df.to_csv("/home/bri/pymol/data/harvey/harvey_master.csv", index=False )

In [9]:
AMBIGUOUS_TARGET= "HSA"   # test

def create_sample_id(row):
    vhh = str(row['vhh'])
    target = str(row['target'])
    residues = str(row['target_residues'])
    
    # Check if target is (ambigous target, meaning the same target has different version --> target residues are needed to be able to identify it, the only one for now is HSA) to apply the specific naming convention
    if target == AMBIGUOUS_TARGET:
        return f"{vhh}_{target}_{residues}"
    else:
        return f"{vhh}_{target}"

# 1. Create the 'sample' column in the master table
master_df['sample'] = master_df.apply(create_sample_id, axis=1)
master_df



,source,type,vhh,vhh_sequence,target,target_residues,target_sequence,binder,KD[M],EC50[M],notes,sample
0,harvey,original,AT118i4h32,EVQLVESGGGLVQPGGSLRLSCAASGYIYRRYRMGWYRQAPGKGRE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,8TH3,AT118i4h32_AGTR1
1,harvey,original,AT201,QVQLQESGGGLVQAGGSLRLSCAASGTISDPPYMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT201_AGTR1
2,harvey,original,AT202,QVQLQESGGGLVQAGGSLRLSCAASGYIFDLPYMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT202_AGTR1
3,harvey,original,AT203,QVQLQESGGLVQAGGSLRLSCAASGYISRSYDMGWYRQAPGKEREL...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT203_AGTR1
4,harvey,original,AT204,QVQLQESGGGLVQAGGSLRLSCAASGYIFTRGFMGWYRQAPGKERE...,AGTR1,1-359,MILNSSTEDGIKRIQDDCPKAGRHNYIFVMIPTLYSIIFVVGIFGN...,1.0,NaN,NaN,NaN,AT204_AGTR1
...,...,...,...,...,...,...,...,...,...,...,...,...
69,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan_nan
70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan_nan
71,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan_nan
72,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nan_nan


In [11]:
master_df3.head()

,source,type,sample,fab,interface,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1000.0,"A,C",1,0.000010
1,snir_ab,original,5ifj_abc_cut,1000.0,"B,C",1,0.000010
2,snir_ab,original,5ig7_abc_cut,10.0,"A,C",1,0.000100
3,snir_ab,original,5ig7_abc_cut,10.0,"B,C",1,0.000100
4,snir_ab,original,5ijk_acx_cut,10.0,"A,C",1,0.000001


In [12]:
master_df4 = master_df3.drop(columns=["fab", "interface"]) #, "Unnamed: 11", "Unnamed: 12", "Unnamed: 13", "Unnamed: 14", "Unnamed: 15", "Unnamed: 16", "Unnamed: 17"])

master_df4

,source,type,sample,binder,KD[M]
0,snir_ab,original,5ifj_abc_cut,1,0.000010
1,snir_ab,original,5ifj_abc_cut,1,0.000010
2,snir_ab,original,5ig7_abc_cut,1,0.000100
3,snir_ab,original,5ig7_abc_cut,1,0.000100
4,snir_ab,original,5ijk_acx_cut,1,0.000001
...,...,...,...,...,...
65,snir_ab,alanine_scan,5ijk_acx_cut_B_Y108A,0,NaN
66,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
67,snir_ab,alanine_scan,5ijk_acx_cut_B_Y31A,0,NaN
68,snir_ab,alanine_scan,5ijk_acx_cut_B_Y38A,0,NaN


### merge

In [13]:
scores_df = pd.merge(master_df4, df_2if, on=['sample'], how='inner')
#df = pd.merge(master_df2, preds_df, on=['sample'], how='inner')
# Display the first few rows to verify
print(scores_df[['sample', 'af3_plddt']])

# Save the merged result
scores_df.to_csv(OUTPUT_DIR /'merged_snir.csv', index=False)

                  sample  af3_plddt
0           5ifj_abc_cut  95.404666
1           5ifj_abc_cut  95.404666
2           5ig7_abc_cut  95.599717
3           5ig7_abc_cut  95.599717
4           5ijk_acx_cut  92.439163
..                   ...        ...
61  5ijk_acx_cut_B_Y108A  91.641826
62   5ijk_acx_cut_B_Y31A  94.546322
63   5ijk_acx_cut_B_Y31A  94.546322
64   5ijk_acx_cut_B_Y38A  91.808649
65   5ijk_acx_cut_B_Y38A  91.808649

[66 rows x 2 columns]
